# 04 — Data Cleaning: 2026 Stack Overflow Survey

Mirrors `02-data-cleaning.ipynb` exactly. Outputs `data/processed/analysis_2026.parquet` with identical column names, dtypes and encodings as `analysis.parquet`.

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from pathlib import Path

RAW_2026  = Path("../data/raw/2026/")
PROCESSED = Path("../data/processed/")
PROCESSED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_2026 / "survey_results_public_2026.csv", low_memory=False)
schema = pd.read_csv(RAW_2026 / "survey_results_schema_2026.csv", encoding="utf-8-sig")
n_raw = df.shape[0]
print(f"Raw responses: {n_raw:,} rows  x  {df.shape[1]} columns")


Raw responses: 30,903 rows  x  212 columns


## Configuration

In [2]:
# 2026 treatment: AISelect is multi-select; AIFreq_1..4 give per-tool frequency.
NOAI_STR   = "I don’t use AI tools"   # curly apostrophe (U+2019)
FREQ_ORDER = {"Daily": 2, "Weekly": 1, "Monthly or Infrequent": 0}
FREQ_COLS  = ["AIFreq_1", "AIFreq_2", "AIFreq_3", "AIFreq_4"]

# Population
PROF_DEV_VALUE     = "I create code as a primary part of my job"   # CodeMain
EMPLOYED_KEYWORDS  = {"Employed", "Independent contractor, freelancer, or self-employed"}

OUTCOME   = "JobSat"
TREATMENT = "AISelect"

ORDINAL_COVARS = ["OrgSize", "Age", "EdLevel"]
NUMERIC_COVARS = ["WorkExp", "YearsCode", "CompTotal"]
NOMINAL_COVARS = ["DevType", "Country", "Industry", "RemoteWork", "ICorPM"]
SECONDARY_COLS = ["AISentiment", "AIComplex", "AITrust"]  # AIThreat absent in 2026

ANALYSIS_COLS = (
    [OUTCOME]
    + [TREATMENT] + FREQ_COLS
    + ["CodeMain", "Employment", "Education", "Role", "WorkSetting", "ICorPM",
       "ConvertedCompYearly"]
    + ["OrgSize", "Age", "Industry", "Country", "WorkExp", "YearsCode"]
    + SECONDARY_COLS
)

# Keep only columns that exist
present = [c for c in ANALYSIS_COLS if c in df.columns]
absent  = [c for c in ANALYSIS_COLS if c not in df.columns]
if absent:
    print(f"WARNING — columns not found: {absent}")
work = df[present].copy()
print(f"Working frame: {work.shape[0]:,} rows  x  {work.shape[1]} columns")


Working frame: 30,903 rows  x  22 columns


## Step 1 — Treatment definition

**Adopter**: selected at least one AI tool AND maximum frequency across `AIFreq_1`–`AIFreq_4` is Daily or Weekly.  
**Non-adopter**: `AISelect` is exactly *“I don’t use AI tools”*.  
**Excluded**: NaN AISelect; contradictory (non-use + tools); tool selected but no frequency given; maximum frequency Monthly or Infrequent only.

In [3]:
# Numeric max frequency per row (2=Daily, 1=Weekly, 0=Monthly/Infrequent, NaN=no freq)
freq_num = work[FREQ_COLS].replace(FREQ_ORDER)
for c in FREQ_COLS:
    freq_num[c] = pd.to_numeric(freq_num[c], errors="coerce")
max_freq = freq_num.max(axis=1)

contains_noai    = work[TREATMENT].str.contains(NOAI_STR, na=False, regex=False)
is_exactly_noai  = (work[TREATMENT] == NOAI_STR)
is_contradictory = contains_noai & ~is_exactly_noai
has_tool_only    = work[TREATMENT].notna() & ~contains_noai

print("Treatment breakdown:")
print(f"  AISelect NaN                : {work[TREATMENT].isna().sum():,}")
print(f"  Contradictory (NOAI+tool)   : {is_contradictory.sum():,}")
print(f"  Tool + no frequency given   : {(has_tool_only & max_freq.isna()).sum():,}")
print(f"  Tool + monthly/infreq only  : {(has_tool_only & (max_freq == 0)).sum():,}")
print(f"  Non-adopters (exact NOAI)   : {is_exactly_noai.sum():,}")
print(f"  Adopters (D/W freq)         : {(has_tool_only & (max_freq >= 1)).sum():,}")

before = len(work)
work["treat_label"]  = "exclude"
work.loc[is_exactly_noai,                 "treat_label"] = "non_adopter"
work.loc[has_tool_only & (max_freq >= 1), "treat_label"] = "adopter"

work = work[work["treat_label"] != "exclude"].reset_index(drop=True).copy()
work["adoption_status"] = work["treat_label"]
work["treated"]         = work["treat_label"].map({"adopter": 1.0, "non_adopter": 0.0})
work["AISelect_daily"]  = work[FREQ_COLS].eq("Daily").any(axis=1).astype(int)

n_after_treat = len(work)
print(f"\nRetained after treatment definition: {n_after_treat:,}  (dropped {before - n_after_treat:,})")
print(work["adoption_status"].value_counts().to_string())


Treatment breakdown:
  AISelect NaN                : 13,439
  Contradictory (NOAI+tool)   : 96
  Tool + no frequency given   : 78
  Tool + monthly/infreq only  : 775
  Non-adopters (exact NOAI)   : 2,907
  Adopters (D/W freq)         : 13,608

Retained after treatment definition: 16,515  (dropped 14,388)
adoption_status
adopter        13608
non_adopter     2907


## Step 2 — Population filters

Same order as `02`: professional developers first (`CodeMain`), then `Employment`.

In [4]:
# Professional-developer filter
before = len(work)
work = work[work["CodeMain"] == PROF_DEV_VALUE].reset_index(drop=True).copy()
n_after_profdev = len(work)
print(f"After professional-developer filter: {n_after_profdev:,} rows  (dropped {before - n_after_profdev:,})")

print(f"\nEmployment categories (post CodeMain):")
print(work["Employment"].value_counts(dropna=False).head(8).to_string())

def is_employed(val):
    if pd.isna(val): return False
    return any(kw in val for kw in EMPLOYED_KEYWORDS)

before = len(work)
work = work[work["Employment"].apply(is_employed)].reset_index(drop=True).copy()
n_after_employment = len(work)
print(f"\nAfter employment filter: {n_after_employment:,} rows  (dropped {before - n_after_employment:,})")

work = work.drop(columns=["CodeMain", "Employment"], errors="ignore")
print(f"Post-population-filter: {len(work):,} rows  x  {work.shape[1]} columns")


After professional-developer filter: 12,918 rows  (dropped 3,597)

Employment categories (post CodeMain):
Employment
Employed                                                9344
Independent contractor, freelancer, or self-employed    1958
Student                                                  818
Not employed                                             586
Retired                                                  123
I prefer not to say                                       89

After employment filter: 11,302 rows  (dropped 1,616)
Post-population-filter: 11,302 rows  x  24 columns


## Step 3 — Outcome

`JobSat` is already numeric (0–10) in 2026; no coercion needed.

In [5]:
print(f"JobSat dtype  : {work['JobSat'].dtype}")
print(f"JobSat range  : {work['JobSat'].min()} – {work['JobSat'].max()}")
print(f"JobSat missing: {work['JobSat'].isna().sum():,}")

before = len(work)
work = work.dropna(subset=["JobSat"]).reset_index(drop=True).copy()
n_after_jobsat = len(work)
print(f"\nAfter requiring non-missing JobSat: {n_after_jobsat:,}  (dropped {before - n_after_jobsat:,})")


JobSat dtype  : float64
JobSat range  : 0.0 – 10.0
JobSat missing: 40

After requiring non-missing JobSat: 11,262  (dropped 40)


## Step 4 — Clean moderators and covariates

In [6]:
# Rename 2026 column names to 2025 names
work = work.rename(columns={
    "Education"  : "EdLevel",
    "Role"       : "DevType",
    "WorkSetting": "RemoteWork",
})
print("Renamed: Education→EdLevel, Role→DevType, WorkSetting→RemoteWork")
print(f"\nDevType top 8:")
print(work["DevType"].value_counts(dropna=False).head(8).to_string())
print(f"\nRemoteWork:")
print(work["RemoteWork"].value_counts(dropna=False).to_string())


Renamed: Education→EdLevel, Role→DevType, WorkSetting→RemoteWork

DevType top 8:
DevType
Developer, full-stack                            4001
Developer, back-end                              1963
Architect, software or solutions                  878
Developer, desktop or enterprise applications     595
Developer, front-end                              445
Developer, embedded applications or devices       426
Developer, mobile                                 359
DevOps engineer or professional                   231

RemoteWork:
RemoteWork
Remote                                                                          4289
Hybrid (some remote, leans heavy to in-person)                                  2190
Hybrid (some in-person, leans heavy to flexibility)                             1805
In-person                                                                       1548
Your choice (very flexible, you can come in when you want or just as needed)    1417
NaN                           

### 4b. Ordinal covariates: `OrgSize`, `Age`, `EdLevel`

`EdLevel` uses **straight apostrophes** (U+0027) in 2026 vs curly in 2025.

In [7]:
ORG_SIZE_ORDER = {
    "Just me - I am a freelancer, sole proprietor, etc.": 0,
    "Less than 20 employees":      1,
    "20 to 99 employees":          2,
    "100 to 499 employees":        3,
    "500 to 999 employees":        4,
    "1,000 to 4,999 employees":    5,
    "5,000 to 9,999 employees":    6,
    "10,000 or more employees":    7,
    # "I don't know" → NaN (intentional)
}

AGE_ORDER = {
    "Under 18 years old":  0,
    "18-24 years old":     1,
    "25-34 years old":     2,
    "35-44 years old":     3,
    "45-54 years old":     4,
    "55-64 years old":     5,
    "65 years or older":   6,
    # "Prefer not to say" → NaN
}

# 2026 uses STRAIGHT apostrophes (U+0027), not curly (U+2019) as in 2025
ED_ORDER_2026 = {
    "Primary/elementary school":                                                           0,
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)":  1,
    "Some college/university study without earning a degree":                              2,
    "Associate degree (A.A., A.S., etc.)":                                                 3,
    "Bachelor's degree (B.A., B.S., B.Eng., etc.)":                                       4,
    "Master's degree (M.A., M.S., M.Eng., MBA, etc.)":                                    5,
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)":                                     6,
    # "Other, please specify" → NaN (intentional)
}

for col, mapping in [("OrgSize", ORG_SIZE_ORDER), ("Age", AGE_ORDER), ("EdLevel", ED_ORDER_2026)]:
    if col not in work.columns:
        print(f"WARNING: {col} not in work"); continue
    mapped    = work[col].map(mapping)
    n_nonull  = work[col].notna().sum()
    n_matched = mapped.notna().sum()
    n_unmatch = n_nonull - n_matched
    print(f"{col:10s}: {n_matched:,} matched,  {n_unmatch:,} non-null unmatched -> NaN")
    if n_unmatch and col == "EdLevel":
        unmatched = work.loc[work[col].notna() & mapped.isna(), col].value_counts()
        for cat, cnt in unmatched.items():
            print(f"  {repr(cat)}: {cnt:,}")
    work[col] = mapped


OrgSize   : 11,065 matched,  189 non-null unmatched -> NaN
Age       : 11,211 matched,  51 non-null unmatched -> NaN
EdLevel   : 11,124 matched,  130 non-null unmatched -> NaN
  'Other, please specify': 130


### 4c. Numeric covariates: `WorkExp`, `YearsCode`

Both are already numeric (float64) in 2026. Clip at 50 and 51 respectively.

In [8]:
work["WorkExp"]   = pd.to_numeric(work["WorkExp"],   errors="coerce").clip(upper=50)
work["YearsCode"] = pd.to_numeric(work["YearsCode"], errors="coerce").clip(upper=51)
print(f"WorkExp  — min={work['WorkExp'].min():.0f}  max={work['WorkExp'].max():.0f}  missing={work['WorkExp'].isna().sum():,}")
print(f"YearsCode — min={work['YearsCode'].min():.0f}  max={work['YearsCode'].max():.0f}  missing={work['YearsCode'].isna().sum():,}")


WorkExp  — min=1  max=50  missing=78
YearsCode — min=1  max=51  missing=71


### 4d. Compensation

`ConvertedCompYearly` trimmed to [1,000, 2,000,000] and log-transformed.

In [9]:
COMP_TRIM_LO, COMP_TRIM_HI = 1_000, 2_000_000
raw_col = "ConvertedCompYearly"
if raw_col in work.columns:
    comp = pd.to_numeric(work[raw_col], errors="coerce")
    comp = comp.where((comp >= COMP_TRIM_LO) & (comp <= COMP_TRIM_HI))
    work["CompLog"]   = np.log1p(comp)
    work["CompTotal"] = comp
    print(f"CompLog: n_valid={work['CompLog'].notna().sum():,}  "
          f"missing={work['CompLog'].isna().sum():,}  ({work['CompLog'].isna().mean():.1%})")
else:
    work["CompLog"]   = np.nan
    work["CompTotal"] = np.nan
    print("CompLog set to NaN (ConvertedCompYearly not found).")

NUMERIC_COVARS_FINAL = ["WorkExp", "YearsCode", "CompLog"]


CompLog: n_valid=8,425  missing=2,837  (25.2%)


### 4e. Nominal covariates

`ICorPM` is collapsed from 4 levels to 2 to match 2025.

In [10]:
ICORPM_MAP = {
    "Individual contributor":                                 "Individual contributor",
    "Individual contributor in a senior or staff-level role": "Individual contributor",
    "People manager":                                         "People manager",
    "Both people manager and individual contributor":         "People manager",
}
work["ICorPM"] = work["ICorPM"].map(ICORPM_MAP)
print("ICorPM after collapse:")
print(work["ICorPM"].value_counts(dropna=False).to_string())

print("\nNominal column null counts:")
for col in NOMINAL_COVARS:
    n = work[col].isna().sum() if col in work.columns else "MISSING"
    print(f"  {col}: {n} NaN")


ICorPM after collapse:
ICorPM
Individual contributor    9595
People manager            1628
NaN                         39

Nominal column null counts:
  DevType: 12 NaN
  Country: 0 NaN
  Industry: 42 NaN
  RemoteWork: 13 NaN
  ICorPM: 39 NaN


## Step 5 — Missingness audit

In [11]:
audit_cols = ORDINAL_COVARS + NUMERIC_COVARS_FINAL + NOMINAL_COVARS
audit = (
    work[audit_cols + ["JobSat", "treated"]]
    .isna().mean().mul(100).round(1)
    .sort_values(ascending=False)
)
print("Missing % per variable (pre-imputation, post-filter):")
print(audit.to_string())
print(f"\nPost-filter rows: {len(work):,}")


Missing % per variable (pre-imputation, post-filter):
CompLog       25.2
OrgSize        1.7
EdLevel        1.2
WorkExp        0.7
YearsCode      0.6
Age            0.5
Industry       0.4
ICorPM         0.3
DevType        0.1
RemoteWork     0.1
Country        0.0
JobSat         0.0
treated        0.0

Post-filter rows: 11,262


> **Decision**: identical missing-data strategy to `02`.  
> Numeric: median imputation + binary `*_missing` indicator.  
> Ordinal: median rank.  Nominal: `"Missing"` level.

In [12]:
# Nominal: NaN -> "Missing"
for col in NOMINAL_COVARS:
    if col not in work.columns: continue
    n = work[col].isna().sum()
    if n:
        work[col] = work[col].fillna("Missing")
        print(f"{col:12s}: filled {n:,} NaN → 'Missing'")

# Ordinal: NaN -> median rank
for col in ORDINAL_COVARS:
    if col not in work.columns: continue
    n = work[col].isna().sum()
    if n:
        med = work[col].median()
        work[col] = work[col].fillna(med)
        print(f"{col:12s}: filled {n:,} NaN → median ({med:.0f})")

# Numeric: median + missingness indicator
indicator_cols = []
for col in NUMERIC_COVARS_FINAL:
    if col not in work.columns: continue
    n = work[col].isna().sum()
    if n:
        ind = f"{col}_missing"
        work[ind]  = work[col].isna().astype(int)
        indicator_cols.append(ind)
        work[col]  = work[col].fillna(work[col].median())
        print(f"{col:12s}: filled {n:,} NaN → median; added {ind}")

# Build final analysis frame
final_cols = (["JobSat", "treated", "AISelect_daily", "adoption_status"]
              + ORDINAL_COVARS
              + NUMERIC_COVARS_FINAL
              + indicator_cols
              + NOMINAL_COVARS)

analysis = work.dropna(subset=["JobSat"])[final_cols].copy()
available_secondary = [c for c in SECONDARY_COLS if c in work.columns]
if available_secondary:
    analysis = analysis.copy()
    analysis[available_secondary] = work.loc[analysis.index, available_secondary].values

n_adopt = (analysis["adoption_status"] == "adopter").sum()
n_ctrl  = (analysis["adoption_status"] == "non_adopter").sum()
print(f"\nFull analysis frame: {len(analysis):,} rows  x  {analysis.shape[1]} columns")
print(f"  Adopters (T=1):     {n_adopt:,}  ({n_adopt/len(analysis):.1%})")
print(f"  Non-adopters (T=0): {n_ctrl:,}  ({n_ctrl/len(analysis):.1%})")


DevType     : filled 12 NaN → 'Missing'
Industry    : filled 42 NaN → 'Missing'
RemoteWork  : filled 13 NaN → 'Missing'
ICorPM      : filled 39 NaN → 'Missing'
OrgSize     : filled 197 NaN → median (3)
Age         : filled 51 NaN → median (3)
EdLevel     : filled 138 NaN → median (4)
WorkExp     : filled 78 NaN → median; added WorkExp_missing
YearsCode   : filled 71 NaN → median; added YearsCode_missing
CompLog     : filled 2,837 NaN → median; added CompLog_missing

Full analysis frame: 11,262 rows  x  21 columns
  Adopters (T=1):     9,783  (86.9%)
  Non-adopters (T=0): 1,479  (13.1%)


## Step 6 — Funnel summary and assertions

In [13]:
print("=" * 68)
print("SAMPLE CONSTRUCTION FUNNEL")
print("=" * 68)
print(f"{'Step':<44} {'Remaining':>9}  {'Dropped':>8}")
print("-" * 68)
print(f"{'Raw survey responses':<44} {n_raw:>9,}  {'--':>8}")
print(f"{'After treatment definition':<44} {n_after_treat:>9,}  {n_raw - n_after_treat:>8,}")
print(f"{'After professional-developer filter':<44} {n_after_profdev:>9,}  {n_after_treat - n_after_profdev:>8,}")
print(f"{'After employment filter':<44} {n_after_employment:>9,}  {n_after_profdev - n_after_employment:>8,}")
print(f"{'After non-missing JobSat':<44} {n_after_jobsat:>9,}  {n_after_employment - n_after_jobsat:>8,}")
print("=" * 68)
print(f"  Adopters      : {n_adopt:,}")
print(f"  Non-adopters  : {n_ctrl:,}")
print()
assert n_after_treat      == 16_515, f"Expected 16,515 after treatment; got {n_after_treat:,}"
assert n_after_profdev    == 12_918, f"Expected 12,918 after profdev; got {n_after_profdev:,}"
assert n_after_employment == 11_302, f"Expected 11,302 after employment; got {n_after_employment:,}"
assert n_after_jobsat     == 11_262, f"Expected 11,262 after JobSat; got {n_after_jobsat:,}"
assert n_adopt            ==  9_783, f"Expected 9,783 adopters; got {n_adopt:,}"
assert n_ctrl             ==  1_479, f"Expected 1,479 non-adopters; got {n_ctrl:,}"
print("✓ All funnel assertions passed")


SAMPLE CONSTRUCTION FUNNEL
Step                                         Remaining   Dropped
--------------------------------------------------------------------
Raw survey responses                            30,903        --
After treatment definition                      16,515    14,388
After professional-developer filter             12,918     3,597
After employment filter                         11,302     1,616
After non-missing JobSat                        11,262        40
  Adopters      : 9,783
  Non-adopters  : 1,479

✓ All funnel assertions passed


## Step 7 — Save

In [14]:
print("Column dtypes:")
print(analysis.dtypes.to_string())


Column dtypes:
JobSat               float64
treated              float64
AISelect_daily         int64
adoption_status       object
OrgSize              float64
Age                  float64
EdLevel              float64
WorkExp              float64
YearsCode            float64
CompLog              float64
WorkExp_missing        int64
YearsCode_missing      int64
CompLog_missing        int64
DevType               object
Country               object
Industry              object
RemoteWork            object
ICorPM                object
AISentiment           object
AIComplex             object
AITrust               object


In [15]:
out_parquet = PROCESSED / "analysis_2026.parquet"
out_csv     = PROCESSED / "analysis_2026.csv"

analysis.to_parquet(out_parquet, index=False)
analysis.to_csv(out_csv, index=False)
print(f"Saved {out_parquet}")
print(f"Saved {out_csv}")
print(f"Final shape: {analysis.shape[0]:,} rows x {analysis.shape[1]} columns")

# Verify column alignment with 2025 parquet (both years' secondary AI cols excluded)
ref = pd.read_parquet(PROCESSED / "analysis.parquet")
SEC_2025 = ["AIThreat", "AISent", "AIAcc", "AIComplex"]  # 2025 secondary cols
cols_2025 = set(ref.columns) - set(SEC_2025)
cols_2026 = set(analysis.columns) - set(available_secondary)
only_2025 = cols_2025 - cols_2026
only_2026 = cols_2026 - cols_2025
if only_2025:
    print(f"\nColumns only in 2025: {sorted(only_2025)}")
if only_2026:
    print(f"Columns only in 2026: {sorted(only_2026)}")
if not only_2025 and not only_2026:
    print("\n✓ Analysis columns match between years (secondary AI cols excluded from both)")
else:
    print("\n⚠ Column mismatch — check above")


Saved ../data/processed/analysis_2026.parquet
Saved ../data/processed/analysis_2026.csv
Final shape: 11,262 rows x 21 columns



✓ Analysis columns match between years (secondary AI cols excluded from both)


## Survey differences from 2025

| Dimension | 2025 | 2026 |
|---|---|---|
| AI treatment question | AISelect single-select with frequency built into answer options | AISelect multi-select (tool types) + AIFreq_1..4 per-tool |
| Non-user string | curly apostrophe (U+2019) | curly apostrophe (U+2019) |
| Plan-to-soon category | Yes (`adoption_status = plan_to_soon`, T=0) | **No** |
| Professional dev filter | MainBranch | CodeMain |
| Education apostrophes | U+2019 (curly) | U+0027 (straight) |
| Role column | multi-select, semicolon-separated | single-select |
| Work setting column | RemoteWork | WorkSetting → renamed to RemoteWork |
| ICorPM levels | 2 | 4 (collapsed to 2) |
| WorkExp / YearsCode | WorkExp numeric; YearsCode string-parsed | numeric floats (clipped) |
| Secondary AI columns | AIThreat, AISent, AIAcc, AIComplex | AISentiment, AIComplex, AITrust |
